# Cell 1 — Markdown cell (not code)

## Schema Understanding — How the 9 Tables Connect

orders is the central table. Every transaction flows through it.
- orders connects to customers via customer_id
- orders connects to order_items via order_id
- order_items is the BRIDGE table connecting orders to products and sellers
- orders connects to payments via order_id (one order can have multiple payment rows)
- orders connects to reviews via order_id
- products connects to category_translation for English names
- geolocation is a lookup table — zip_code_prefix is the connection key

In [1]:
# Cell 2 — Verify Primary Keys

import pandas as pd

# Load tables (copy your loading code from notebook 01)
DATA_PATH = "../data/raw/"
df_orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
df_customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
df_products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
df_order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
df_payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
df_sellers = pd.read_csv(DATA_PATH + "olist_sellers_dataset.csv")

print(" PRIMARY KEY VERIFICATION......... \n")

# Check if order_id is truly unique in orders table
total = len(df_orders)
unique = df_orders['order_id'].nunique()
print(f"orders — total rows: {total:,} | unique order_id: {unique:,}", end="  ")
print("TRUE PRIMARY KEY" if total == unique else " DUPLICATES EXIST")

# Check customer_id in customers
total = len(df_customers)
unique = df_customers['customer_id'].nunique()
print(f"customers — total rows: {total:,} | unique customer_id: {unique:,}", end="  ")
print("TRUE PRIMARY KEY" if total == unique else "DUPLICATES EXIST")

# Check product_id in products
total = len(df_products)
unique = df_products['product_id'].nunique()
print(f"products — total rows: {total:,} | unique product_id: {unique:,}", end="  ")
print("TRUE PRIMARY KEY" if total == unique else "DUPLICATES EXIST")

# Check seller_id in sellers
total = len(df_sellers)
unique = df_sellers['seller_id'].nunique()
print(f"sellers — total rows: {total:,} | unique seller_id: {unique:,}", end="  ")
print("TRUE PRIMARY KEY" if total == unique else "DUPLICATES EXIST")

 PRIMARY KEY VERIFICATION......... 

orders — total rows: 99,441 | unique order_id: 99,441  TRUE PRIMARY KEY
customers — total rows: 99,441 | unique customer_id: 99,441  TRUE PRIMARY KEY
products — total rows: 32,951 | unique product_id: 32,951  TRUE PRIMARY KEY
sellers — total rows: 3,095 | unique seller_id: 3,095  TRUE PRIMARY KEY


In [2]:
# Cell 3 — Verify Foreign Key behaviour in order_items

total = len(df_order_items)
unique_orders = df_order_items['order_id'].nunique()
print(f"order_items total rows:    {total:,}")
print(f"unique order_ids:          {unique_orders:,}")
print(f"average items per order:   {total/unique_orders:.2f}")
print()
print("order_id in order_items is NOT a primary key")
print("It is a FOREIGN KEY — one order can appear multiple times")

order_items total rows:    112,650
unique order_ids:          98,666
average items per order:   1.14

order_id in order_items is NOT a primary key
It is a FOREIGN KEY — one order can appear multiple times


In [3]:
# Cell 4 — Merge 1: orders + customers
# Business purpose: to analyse orders BY customer location

print("MERGE 1: orders + customers")
print(f"Before — orders: {len(df_orders):,} rows")

df_orders_customers = pd.merge(
    df_orders,
    df_customers,
    on='customer_id',
    how='left'   # keep ALL orders even if customer not found
)

print(f"After  — merged: {len(df_orders_customers):,} rows")
print(f"Nulls in customer_state after merge: {df_orders_customers['customer_state'].isnull().sum()}")
print("Row count preserved — merge is clean\n" if len(df_orders_customers)==len(df_orders) else "Row count changed — investigate")

MERGE 1: orders + customers
Before — orders: 99,441 rows
After  — merged: 99,441 rows
Nulls in customer_state after merge: 0
Row count preserved — merge is clean



In [4]:
# Cell 7 — Business Question 1
# Q: Which state has the most customers?

customers_by_state = df_customers['customer_state'].value_counts().head(10)
print("Q1: Top 10 states by number of customers")
print(customers_by_state)
print()
print("Business Insight: [write your observation here]")

Q1: Top 10 states by number of customers
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

Business Insight: [write your observation here]


In [5]:
# Cell 8 — Business Question 2
# Q: How many orders does each customer place on average?
# (Hint: customer_id vs customer_unique_id — they are different!)

total_orders = len(df_orders)
unique_customers = df_customers['customer_unique_id'].nunique()
avg_orders = total_orders / unique_customers

print("Q2: Average orders per customer")
print(f"Total orders:          {total_orders:,}")
print(f"Unique customers:      {unique_customers:,}")
print(f"Average orders/customer: {avg_orders:.2f}")
print()
print("Business Insight: [what does this tell you about customer loyalty?]")

Q2: Average orders per customer
Total orders:          99,441
Unique customers:      96,096
Average orders/customer: 1.03

Business Insight: [what does this tell you about customer loyalty?]


In [6]:
import pandas as pd
# merge order_items with products first to get category name
df_items_products = pd.merge(
    df_order_items,
    df_products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

category_orders = df_items_products.groupby('product_category_name')['order_id'].nunique()
category_orders = category_orders.sort_values(ascending=False).head(5)
print("Q3: Top 5 categories by number of orders")
print(category_orders)

Q3: Top 5 categories by number of orders
product_category_name
cama_mesa_banho           9417
beleza_saude              8836
esporte_lazer             7720
informatica_acessorios    6689
moveis_decoracao          6449
Name: order_id, dtype: int64


# Final Markdown Cell

## Day 2 Summary — Schema Understanding Complete

### Key Connections Verified
 orders ↔ customers: via customer_id 
 orders ↔ order_items: via order_id (1 order = multiple items)
 order_items ↔ products: via product_id 
 order_items ↔ sellers: via seller_id 
 orders ↔ payments: via order_id (1 order = multiple payment rows)
 orders ↔ reviews: via order_id

### Primary Keys Verified
 order_id in orders: UNIQUE 
 customer_id in customers: UNIQUE 
 product_id in products: UNIQUE 
 seller_id in sellers: UNIQUE 

### 3 Business Questions Answered
1. **question 1 result understanding**
    SP,RJ,MG are the top 3 states with highest number of cudtomers and SP alone has 41,746 customers that's 42% of your entire customer base (41,746 / 96,096).RJ and MG combined add another 25%.Together, just 3 out of 27 Brazilian states make up 67% of all customers.
2. **question 2 result understanding**
    using mean as parameter , every customer oreders atleast 1 item per order.After spending more time , i can finally conclude that it is "orders per customer" not "items per order"
    Correct interpretation:
     Average orders per customer = 1.03
     This means repeat purchase rate is approximately 3%
     97% of customers in this dataset placed exactly ONE order and never returned
     This is a customer retention problem, not a basket-size observation
     This finding directly motivates the cohort/RFM analysis planned for Week 3
3. **question 3 result understanding**
    Top category (cama_mesa_banho / Bed, Bath & Table) generates 9,417 orders
    roughly 13% more orders than the #2 category (Health & Beauty).

This indicates Olist's customer base is weighted toward home/lifestyle 
products rather than electronics or fashion, which is common for 
Brazilian e-commerce platforms. 

Action item for Week 3: merge category_translation table to get English 
names before any chart or dashboard goes external-facing.

### What I can now do
 Build any multi-table merge to answer a business question
 Trace the path from any question to the tables required
 Verify joins produce the correct number of rows